# Phase 2 — VAE Augmentation & Augmented Model Training
**DiagGen+ | Advanced AI Group Project 2026**

## Objectives
1. Load Phase 1 baseline metrics and rare class definitions.
2. Generate symptom embeddings for the training set.
3. Train a Conditional VAE on the rare-class embeddings.
4. Generate synthetic samples for each rare class until target count reached.
5. Validate synthetic samples statistically.
6. Retrain BERT on the augmented dataset.
7. Compare augmented vs baseline metrics — plot the improvement.

## Experiment Design
Both models are evaluated on the **same held-out test set** (`test_imbalanced.csv`).
The only difference between them is the training data:

| Model | Training input | Rare class samples |
|-------|---------------|-------------------|
| Baseline (Phase 1) | `train_imbalanced.csv` | ~20 per class |
| Augmented (Phase 2) | `train_imbalanced.csv` + VAE synthetic | ~250 per class |

> **Gate 2 complete** when: augmented model outperforms baseline on rare-class F1, comparison charts saved.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path

from src.utils.config_loader import cfg, get
from src.utils.logger import get_logger
from src.preprocessing.cleaner import preprocess_batch
from src.preprocessing.embeddings import embeddings as emb_model
from src.models.vae import ConditionalVAE
from src.models.classifier import DiagnosisClassifier
from src.training.trainer import load_processed_splits, build_label_maps
from src.training.augment import (
    identify_rare_classes, train_vae,
    generate_synthetic_samples, validate_synthetic, smote_augment
)
from src.evaluation.metrics import evaluate, plot_confusion_matrix, plot_class_f1_comparison

logger = get_logger('phase2_vae')
FIGURES_DIR   = Path('../reports/figures')
PROCESSED_DIR = Path('../data/processed')
SYNTHETIC_DIR = Path('../data/synthetic')
SYNTHETIC_DIR.mkdir(parents=True, exist_ok=True)
print('Imports OK')

## 1. Load Phase 1 Context

In [ ]:
# Load rare class definitions from Phase 0
rare_meta     = json.loads((PROCESSED_DIR / 'rare_classes.json').read_text())
RARE_DISEASES = rare_meta['rare_diseases']
RARE_N        = rare_meta['rare_n_samples']
print(f'Rare classes ({len(RARE_DISEASES)}): {RARE_DISEASES}')
print(f'Samples per rare class (before augmentation): {RARE_N}')

# Load baseline metrics from Phase 1
baseline = json.loads((PROCESSED_DIR / 'baseline_metrics.json').read_text())
print(f"\nBaseline Macro F1:          {baseline['macro_f1']:.4f}")
print(f"Baseline rare class F1:     {baseline['rare_classes_mean_f1']:.4f}")
print(f"Baseline common class F1:   {baseline['common_classes_mean_f1']:.4f}")
print('\nPer-class F1 (baseline):')
for disease, f1 in baseline['per_class_f1'].items():
    marker = ' ← RARE' if disease in RARE_DISEASES else ''
    print(f'  {disease:<40} {f1:.4f}{marker}')

## 2. Generate Symptom Embeddings

In [ ]:
# Load training split
train_df, val_df, test_df = load_processed_splits()
all_df = pd.concat([train_df, val_df, test_df])
label2id, id2label, le = build_label_maps(all_df)

# Prepare text
def get_texts(df):
    if 'symptoms_clean' in df.columns:
        return df['symptoms_clean'].fillna('').tolist()
    return preprocess_batch(df['symptoms'].tolist())

train_texts  = get_texts(train_df)
train_labels_enc = le.transform(train_df['disease'].tolist())
print(f'Training samples: {len(train_texts)}')

In [ ]:
# Load embedding model (downloads ~100MB on first run)
print(f'Loading {get("nlp.embeddings.type")} embeddings...')
emb_model.load()
print(f'Embedding dim: {emb_model.dim}')

# Embed all training samples
print('Generating sentence embeddings for training set...')
train_embeddings = np.array([
    emb_model.embed_sentence(text.split())
    for text in train_texts
])
print(f'Embedding matrix shape: {train_embeddings.shape}')

# Save embeddings for later use
np.save(PROCESSED_DIR / 'train_embeddings.npy', train_embeddings)
np.save(PROCESSED_DIR / 'train_labels_enc.npy', train_labels_enc)
print('Embeddings saved.')

## 3. Train the Conditional VAE

In [ ]:
# Isolate rare-class samples for VAE training
# (VAE only needs to learn the distribution of rare-class embeddings)
rare_mask    = np.array([id2label[i] in RARE_DISEASES for i in train_labels_enc])
rare_embs    = train_embeddings[rare_mask]
rare_labels  = train_labels_enc[rare_mask]

print(f'VAE training samples (rare classes only): {len(rare_embs)}')
print(f'Input dim: {rare_embs.shape[1]} | Num classes: {len(label2id)}')

# Note: we train VAE on rare classes only to keep the learned distribution tight.
# If you want to also generate common-class augmentation, pass all embeddings here.

In [ ]:
# Train VAE
# Expected runtime: 5–15 minutes on CPU
print(f'Training Conditional VAE...')
print(f'  Epochs:     {get("vae.training.num_epochs")}')
print(f'  Latent dim: {get("vae.latent_dim")}')
print(f'  Hidden dims:{get("vae.hidden_dims")}')
print()

vae = train_vae(
    embeddings=rare_embs,
    labels=rare_labels,
    num_classes=len(label2id)
)
print('\n✅ VAE training complete. Checkpoint saved to models/vae/cvae.pt')

## 4. Generate & Validate Synthetic Samples

If the VAE loss did not converge (still high after 50 epochs), skip to the SMOTE fallback cell.

In [ ]:
TARGET_PER_CLASS = get('vae.augmentation.target_samples_per_rare_class', 250)
existing_counts  = dict(train_df['disease'].value_counts())

rare_class_ids = [
    label2id[d] for d in RARE_DISEASES if d in label2id
]

synthetic_df = generate_synthetic_samples(
    vae=vae,
    rare_class_ids=rare_class_ids,
    id2label=id2label,
    existing_counts=existing_counts
)

print(f'Synthetic records generated: {len(synthetic_df)}')
print(synthetic_df['disease'].value_counts())

In [ ]:
# Statistical validation: compare real vs synthetic embedding distributions
synthetic_embeddings = np.array([
    np.array(row['embedding']) for _, row in synthetic_df.iterrows()
])

validation = validate_synthetic(rare_embs, synthetic_embeddings)
print(f'Validation result: {"PASS" if validation["pass"] else "FAIL"}')
print(f'  Mean absolute diff: {validation["mean_absolute_diff"]:.4f}')
print(f'  Std absolute diff:  {validation["std_absolute_diff"]:.4f}')

if not validation['pass']:
    print('\n⚠️  Validation failed. Consider:')
    print('   1. Training the VAE for more epochs (increase vae.training.num_epochs in config.yaml)')
    print('   2. Using the SMOTE fallback in the next cell')

In [ ]:
# ── SMOTE fallback (run this cell ONLY if VAE validation failed) ──────────────
# Comment out this cell if VAE validation passed.

USE_SMOTE_FALLBACK = False   # set True if VAE failed validation

if USE_SMOTE_FALLBACK:
    print('Using SMOTE fallback...')
    aug_embeddings, aug_labels = smote_augment(rare_embs, rare_labels)
    print(f'After SMOTE: {len(aug_embeddings)} samples (from {len(rare_embs)})')
    # Build synthetic_df from SMOTE output
    smote_rows = []
    for emb, lbl in zip(aug_embeddings[len(rare_embs):], aug_labels[len(rare_labels):]):
        smote_rows.append({'disease': id2label[lbl], 'is_synthetic': True, 'embedding': emb.tolist()})
    synthetic_df = pd.DataFrame(smote_rows)
    print(f'SMOTE synthetic records: {len(synthetic_df)}')
else:
    print('Using VAE-generated samples (SMOTE fallback not needed).')

In [ ]:
# Save synthetic records for reproducibility and report appendix
synthetic_df_save = synthetic_df[['disease','is_synthetic']].copy()
synthetic_df_save.to_csv(SYNTHETIC_DIR / 'synthetic_rare_records.csv', index=False)
np.save(SYNTHETIC_DIR / 'synthetic_embeddings.npy', synthetic_embeddings)
print(f'Synthetic data saved to data/synthetic/')
print(f'  synthetic_rare_records.csv  — metadata (disease labels)')
print(f'  synthetic_embeddings.npy    — embedding vectors for augmented training')

## 5. Reconstruct Augmented Text for BERT Retraining

In [ ]:
# The VAE generates embedding vectors, not raw text.
# To retrain BERT, we need text. Strategy: for each synthetic embedding,
# find the K nearest real training samples and slightly rephrase the closest one.
# Simpler alternative used here: nearest-neighbour retrieval (no LLM cost).

from sklearn.metrics.pairwise import cosine_similarity

def embedding_to_text(syn_embedding: np.ndarray, real_embeddings: np.ndarray,
                      real_texts: list[str], k: int = 1) -> str:
    """Return the real text whose embedding is closest to the synthetic one."""
    sims = cosine_similarity([syn_embedding], real_embeddings)[0]
    idx  = np.argmax(sims)
    return real_texts[idx]

# Build a lookup of real embeddings per rare class
augmented_texts  = []
augmented_labels = []

for _, row in synthetic_df.iterrows():
    disease  = row['disease']
    syn_emb  = np.array(row['embedding'])
    cls_id   = label2id[disease]

    # Real embeddings for this class
    mask      = train_labels_enc == cls_id
    real_embs = train_embeddings[mask]
    real_txts = [train_texts[i] for i in np.where(mask)[0]]

    proxy_text = embedding_to_text(syn_emb, real_embs, real_txts)
    augmented_texts.append(proxy_text)
    augmented_labels.append(cls_id)

print(f'Augmented text samples generated: {len(augmented_texts)}')
print('Sample synthetic text:', augmented_texts[0][:100] if augmented_texts else 'None')

In [ ]:
# Combine original training set + synthetic augmentation
aug_train_texts  = train_texts  + augmented_texts
aug_train_labels = list(train_labels_enc) + augmented_labels

print(f'Original training size:  {len(train_texts)}')
print(f'After augmentation:      {len(aug_train_texts)}')
print(f'New rare class counts:')
from collections import Counter
counts = Counter([id2label[l] for l in aug_train_labels])
for d in RARE_DISEASES:
    print(f'  {d:<40} {counts.get(d, 0)}')

## 6. Retrain BERT on Augmented Dataset

In [ ]:
# Train augmented model
# Note: saves to models/bert/ — this OVERWRITES the baseline checkpoint.
# If you want to keep both, change bert.output_dir in config.yaml first,
# e.g. set it to 'models/bert_augmented' before running this cell.

val_texts   = get_texts(val_df)
val_labels_enc = le.transform(val_df['disease'].tolist())

clf_aug   = DiagnosisClassifier(label2id, id2label)
train_ds  = clf_aug.prepare_dataset(aug_train_texts, aug_train_labels)
val_ds    = clf_aug.prepare_dataset(val_texts, val_labels_enc.tolist())

clf_aug.train(train_ds, val_ds)
print('\n✅ Augmented model training complete.')

## 7. Evaluate & Compare

In [ ]:
from tqdm import tqdm

test_texts_list = get_texts(test_df)
test_labels_enc = le.transform(test_df['disease'].tolist())
label_names     = [id2label[i] for i in range(len(id2label))]

print('Running inference on test set (augmented model)...')
y_pred_aug = [
    label2id[clf_aug.predict(text)['top_disease']]
    for text in tqdm(test_texts_list)
]

augmented_metrics = evaluate(test_labels_enc.tolist(), y_pred_aug, label_names)
print(f"Augmented Accuracy:  {augmented_metrics['accuracy']:.4f}")
print(f"Augmented Macro F1:  {augmented_metrics['macro_f1']:.4f}")

In [ ]:
# Per-class comparison
import pandas as pd
aug_report   = pd.DataFrame(augmented_metrics['report']).T.loc[label_names]
aug_rare_f1  = aug_report[aug_report.index.isin(RARE_DISEASES)]['f1-score'].mean()
aug_comm_f1  = aug_report[~aug_report.index.isin(RARE_DISEASES)]['f1-score'].mean()

print('\n── Comparison Summary ──────────────────────────────────────────────')
print(f'{"Metric":<35} {"Baseline":>10} {"Augmented":>10} {"Delta":>10}')
print('-' * 70)
print(f'{"Accuracy":<35} {baseline["accuracy"]:>10.4f} {augmented_metrics["accuracy"]:>10.4f} {augmented_metrics["accuracy"]-baseline["accuracy"]:>+10.4f}')
print(f'{"Macro F1":<35} {baseline["macro_f1"]:>10.4f} {augmented_metrics["macro_f1"]:>10.4f} {augmented_metrics["macro_f1"]-baseline["macro_f1"]:>+10.4f}')
print(f'{"Rare class mean F1":<35} {baseline["rare_classes_mean_f1"]:>10.4f} {aug_rare_f1:>10.4f} {aug_rare_f1-baseline["rare_classes_mean_f1"]:>+10.4f}')
print(f'{"Common class mean F1":<35} {baseline["common_classes_mean_f1"]:>10.4f} {aug_comm_f1:>10.4f} {aug_comm_f1-baseline["common_classes_mean_f1"]:>+10.4f}')
print()
print('This table goes directly into the report Results section as Table 2.')

In [ ]:
# Side-by-side per-class F1 comparison chart
plot_class_f1_comparison(
    baseline=baseline,
    augmented=augmented_metrics,
    label_names=label_names,
    save_path=FIGURES_DIR / 'f1_comparison_baseline_vs_augmented.png'
)
print('Comparison chart saved — use in report Results section.')

In [ ]:
# Save augmented metrics for evaluation notebook
(PROCESSED_DIR / 'augmented_metrics.json').write_text(
    json.dumps({
        'accuracy':    augmented_metrics['accuracy'],
        'macro_f1':    augmented_metrics['macro_f1'],
        'weighted_f1': augmented_metrics['weighted_f1'],
        'per_class_f1': {
            n: float(aug_report.loc[n, 'f1-score'])
            for n in label_names if n in aug_report.index
        },
        'rare_classes_mean_f1':   float(aug_rare_f1),
        'common_classes_mean_f1': float(aug_comm_f1),
    }, indent=2)
)
print('Augmented metrics saved to data/processed/augmented_metrics.json')

## Phase 2 Summary ✅ — Gate 2 Reached

Fill in after running:

| Metric | Baseline | Augmented | Delta |
|--------|----------|-----------|-------|
| Accuracy | — | — | — |
| Macro F1 | — | — | — |
| Rare class mean F1 | — | — | — |
| Common class mean F1 | — | — | — |

**All 5 required techniques are now active:**
- ✅ NLP (spaCy + GloVe)
- ✅ Transformers/LLMs (BioBERT fine-tuning)
- ✅ Generative AI (Conditional VAE)
- ✅ Model Training Approaches (transfer learning + data augmentation)
- ✅ Prompt Engineering (chain-of-thought explanations in app)

> **If Phase 3 cannot be completed:** submit at Gate 2 with this notebook, Phase 1 notebook, the Streamlit app, and the comparison charts. That is a complete, high-quality submission.